In [1]:
import os
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from quiet import silence
silence()  # turn down some noisy library logging so the agent's trace is easy to read

os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
load_dotenv(override=True)

True

In [2]:
MODEL = "gemini-3.1-flash-lite"

agent = LlmAgent(
    model=MODEL,
    name="assistant",
    instruction="You are a concise, friendly assistant. Reply in a single short sentence."
)

In [5]:
result = await InMemoryRunner(agent=agent).run_debug("Say hello in Spanish", verbose=True)

assistant > ¡Hola!


In [6]:
import board

board.reset_board()
board.add_goal("read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt',
  'status': 'pending',
  'result': ''}]

In [7]:
board.show_board()

Goal #1: read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt

In [8]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}


In [9]:
board_agent = LlmAgent(
    model=MODEL,
    name="board_agent",
    instruction="You are manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [10]:
result = await InMemoryRunner(agent=board_agent).run_debug(
    "What is on the board right now, and what is its status?",
    verbose=True)

board_agent > [Calling tool: show_todos({})]
board_agent > [Tool result: {'result': [{'id': 1, 'parent_id': None, 'task': 'read notes.txt, translate its contents into natura...]
board_agent > On the board, there is one pending task:

*   **Task:** "read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt"
*   **Status:** Pending


In [11]:
import requests

def send_push_notification(message: str) -> str:
    """Send a short push notification to the user's phone to report a result or that a job is done."""
    payload = {
        "token": os.getenv("PUSHOVER_TOKEN"),
        "user": os.getenv("PUSHOVER_USER"),
        "message": message,
        "sound": "climb"
    }
    response = requests.post("https://api.pushover.net/1/messages.json", data=payload)
    return f"Push notification sent (status {response.status_code})."

In [15]:
notifier = LlmAgent(
    model=MODEL,
    name="notifier",
    instruction="You notify the user. When asked, use your tool to send a push notification.",
    tools=[send_push_notification],
)

result = await InMemoryRunner(agent=notifier).run_debug(
    "Send a push notification that says hello from Google ADK.",
    verbose=True)

notifier > [Calling tool: send_push_notification({'message': 'hello from Google ADK.'})]
notifier > [Tool result: {'result': 'Push notification sent (status 200).'}]
notifier > The push notification "hello from Google ADK." has been sent successfully.


In [16]:
import subprocess
from pathlib import Path
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

workspace = Path("task_worker/workspace").resolve()   # the only folder the agent may touch

filesystem = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        ),
        timeout=60
    ),
    errlog=subprocess.DEVNULL,
)

In [17]:
file_agent = LlmAgent(
    model=MODEL,
    name="file_agent",
    instruction="you can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [18]:
result = await InMemoryRunner(agent=file_agent).run_debug(
    "Read notes.txt and summarize it in one short sentence.",
    verbose=True)

file_agent > [Calling tool: list_allowed_directories({})]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Allowed directories:\n/home/hieunv/dev/agent_llm/agentic_cour...]
file_agent > [Calling tool: list_directory({'path': '/home/hieunv/dev/agent_llm/agentic_cours...)]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': '[FILE] notes.txt'}], 'structuredContent': {'content': '[FILE]...]
file_agent > [Calling tool: read_text_file({'path': '/home/hieunv/dev/agent_llm/agentic_cours...)]
file_agent > [Tool result: {'content': [{'type': 'text', 'text': 'Welcome to the team.\n\nToday we are building a small languag...]
file_agent > The notes outline a collaborative process for building a language tutor where team members complete individual, assigned tasks from a shared board.


In [19]:
from google.genai import types

INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = LlmAgent(
    model=MODEL,
    name="task_worker",
    instruction=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem, send_push_notification],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.")
board.claim_todo(goal_id)

# Drive the loop ourselves with run_async instead of run_debug, so we can watch it:
# print each tool call, and re-draw the board whenever a step is planned or ticked off.
runner = InMemoryRunner(agent=worker)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="ed")

async for event in runner.run_async(
    user_id="ed",
    session_id=session.id,
    new_message=types.UserContent("Please work the pending goal on the board."),
):
    for call in event.get_function_calls():
        print(f"tool: {call.name}({call.args})")
    if any(r.name in ("plan_steps", "complete_task") for r in event.get_function_responses()):
        board.show_board()
    if event.is_final_response():
        print(event.content.parts[0].text)

tool: show_todos({})
tool: list_directory({'path': '.'})
tool: plan_steps({'goal_id': 1, 'steps': ['Read the contents of notes.txt.', 'Translate the contents into natural Spanish.', 'Write the translated Spanish to a new file named spanish.txt.', 'Send a push notification with a summary of the task completion.']})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.
  Step #2: Read the contents of notes.txt.
  Step #3: Translate the contents into natural Spanish.
  Step #4: Write the translated Spanish to a new file named spanish.txt.
  Step #5: Send a push notification with a summary of the task completion.

tool: read_text_file({'path': 'notes.txt'})
tool: complete_task({'result': 'Successfully read notes.txt.', 'task_id': 2})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.
  Step #2: Read the contents of notes.txt.  Successfully read notes.txt.
  Step #3: Translate the contents into natural Spanish.
  Step #4: Write the translated Spanish to a new file named spanish.txt.
  Step #5: Send a push notification with a summary of the task completion.

tool: complete_task({'task_id': 3, 'result': 'Translated the text to Spanish.'})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.
  Step #2: Read the contents of notes.txt.  Successfully read notes.txt.
  Step #3: Translate the contents into natural Spanish.  Translated the text to Spanish.
  Step #4: Write the translated Spanish to a new file named spanish.txt.
  Step #5: Send a push notification with a summary of the task completion.

... (3 outputs left out)

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.  Task completed: notes.txt read, translated to Spanish, saved to spanish.txt, and notified.
  Step #2: Read the contents of notes.txt.  Successfully read notes.txt.
  Step #3: Translate the contents into natural Spanish.  Translated the text to Spanish.
  Step #4: Write the translated Spanish to a new file named spanish.txt.  Wrote translated Spanish to spanish.txt.
  Step #5: Send a push notification with a summary of the task completion.

tool: complete_task({'task_id': 5, 'result': 'Sent push notification.'})


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt and send a push notification with a summary.  Task completed: notes.txt read, translated to Spanish, saved to spanish.txt, and notified.
  Step #2: Read the contents of notes.txt.  Successfully read notes.txt.
  Step #3: Translate the contents into natural Spanish.  Translated the text to Spanish.
  Step #4: Write the translated Spanish to a new file named spanish.txt.  Wrote translated Spanish to spanish.txt.
  Step #5: Send a push notification with a summary of the task completion.  Sent push notification.

The pending goal has been completed. 

1.  Read `notes.txt`.
2.  Translated the content into Spanish.
3.  Saved the translation to `spanish.txt`.
4.  Sent a push notification confirming completion.
